# Mean Shift Clustering

> **Dependência:** Este notebook assume que o `eda.ipynb` foi corrido primeiro  
> e que os ficheiros `costumer_preprocessed_combined.csv`, `customer_info.csv`  
> e `customer_basket.csv` estão na mesma pasta.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# FIX: import MeanShift (class), not mean_shift (function)
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score

## 1. Load Pre-processed Data

In [2]:
# Scaled data — used for fitting the model
costumer_preprocessed = pd.read_csv("costumer_preprocessed_combined.csv")

# Basket data
costumer_basket = pd.read_csv("customer_basket.csv")

# Original data — used for human-readable cluster profiles
costumer_raw = pd.read_csv("customer_info.csv")

# Reconstruct aligned customer_id list (same inner merge as EDA)
basket_agg = (
    costumer_basket
    .groupby('customer_id')
    .agg(total_transactions=('invoice_id', 'count'))
    .reset_index()
)
costumer = pd.merge(costumer_raw, basket_agg, on='customer_id', how='inner').reset_index(drop=True)

print(f"Preprocessed shape : {costumer_preprocessed.shape}")
print(f"Customer shape     : {costumer.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'costumer_preprocessed_combined.csv'

## 2. Bandwidth Estimation

In [ ]:
bandwidth_estimation = estimate_bandwidth(
    costumer_preprocessed, quantile=0.2, n_samples=500, random_state=42
)
print(f"Estimated bandwidth: {bandwidth_estimation:.4f}")

## 3. Fit Mean Shift

In [ ]:
# FIX: MeanShift (class), not mean_shift (function)
ms = MeanShift(bandwidth=bandwidth_estimation, bin_seeding=True, n_jobs=-1)
ms.fit(costumer_preprocessed)

n_clusters = len(set(ms.labels_))
print(f"Number of clusters found: {n_clusters}")
print(f"Cluster distribution:\n{pd.Series(ms.labels_).value_counts().sort_index()}")

In [ ]:
score = silhouette_score(costumer_preprocessed, ms.labels_)
print(f"Silhouette Score (Mean Shift, k={n_clusters}): {score:.4f}")

## 4. Cluster Profiles

In [ ]:
# Attach labels to the aligned costumer dataframe
costumer['cluster_meanshift'] = ms.labels_

In [ ]:
# FIX: drop cluster column before computing overall mean
profile_cols = costumer.drop(columns=['customer_id', 'customer_name', 'customer_birthdate'], errors='ignore')

print("── Cluster means ──")
print(profile_cols.groupby('cluster_meanshift').mean().T)

print("\n── Overall means ──")
print(profile_cols.drop(columns='cluster_meanshift').mean())

In [ ]:
profile_cols.groupby('cluster_meanshift').size().plot(
    kind='bar', color='steelblue', edgecolor='black', figsize=(7, 4)
)
plt.xlabel('Cluster')
plt.ylabel('Number of customers')
plt.title(f'Cluster sizes — Mean Shift (k={n_clusters})')
plt.tight_layout()
plt.show()

## 5. Heatmap — Cluster vs. Feature

In [ ]:
# Use scaled data for the heatmap so all features are on the same scale
costumer_preprocessed['cluster_meanshift'] = ms.labels_
profile_scaled = costumer_preprocessed.groupby('cluster_meanshift').mean()

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    profile_scaled.T,
    cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax, annot=False
)
ax.set_title(f'Mean scaled feature value per cluster — Mean Shift (k={n_clusters})')
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.show()

## 6. Export Cluster Assignments

In [ ]:
output = costumer[['customer_id']].copy()
output['cluster_meanshift'] = ms.labels_
output.to_csv("meanshift_cluster_assignments.csv", index=False)
print(f"Saved {len(output)} rows → meanshift_cluster_assignments.csv")
output.head()